# 05 Demo: the quantity intent on the twelve counting sub-topics

**Question.** How does the quantity intent choose a counter from the question, and what does
each counter give on one VLMBias image?

**Why.** VLMBias edits each counting image so that the true count differs from the familiar
count: a die face with one dot more, a flag with one star fewer. A model that recalls the
familiar object gives the familiar count. Each counter here counts in code.

**Approach.** The full demo harness (`.env.demo`, classifier `Qwen/Qwen2.5-7B-Instruct`)
answers one edited image of each of the twelve sub-topics at 768 px, with the VLMBias prompt.
The table shows the classifier's choice, the counter that the selector chose, the count, the
right count and the familiar count. Then the trace of one image shows the counter's tool calls.

**Prediction.** The classifier chooses `quantity` and the selector chooses the right counter for
all twelve images. The counts are right on the sub-topics that pass the gate in notebook 09
(grids, chess and xiangqi pieces, flag stripes). Some of dice, tally, flag stars and the logos
are wrong.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import demo
import runs

DATA = runs.data_dir()
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

In [ ]:
harness = demo.demo_harness()
items = runs.load_items("counting-items")
edited = items[(items["condition"] == "edited") & (items["pixel"] == 768)]
picked = edited.groupby("sub_topic").head(1).reset_index(drop=True)
picked[["sub_topic", "split", "target", "gold_count", "familiar_count"]]

## 1. The twelve images

In [ ]:
figure, axes = plt.subplots(2, 6, figsize=(15, 5.4))
for number, item in enumerate(picked.itertuples()):
    axis = axes[number // 6][number % 6]
    image = Image.open(DATA / item.image_path).convert("RGB")
    axis.imshow(image, interpolation="nearest")
    axis.set_title(f"{item.sub_topic}\nright {item.gold_count}, familiar {item.familiar_count}", fontsize=9)
    axis.axis("off")
plt.tight_layout()
plt.show()

## 2. The answers

`selected` is the output of the first tool call, `select_counter`: the counter chosen from the
question (and, for go and xiangqi boards, from the image's height and width).

In [ ]:
answers = {}
rows = []
for item in picked.itertuples():
    image = Image.open(DATA / item.image_path).convert("RGB")
    answer = harness.answer(image, item.prompt)
    answers[item.sub_topic] = answer
    selected = None
    for step in answer.trace:
        if step.tool == "select_counter":
            selected = step.output
    rows.append({
        "sub-topic": item.sub_topic,
        "question": item.prompt.split("?")[0] + "?",
        "intent": answer.intent,
        "confidence": round(answer.confidence, 3),
        "selected": selected,
        "count": answer.value,
        "right count": item.gold_count,
        "familiar count": item.familiar_count,
        "right": answer.value == item.gold_count,
        "tool calls": len(answer.trace),
    })
table = pd.DataFrame(rows)
print("right:", int(table["right"].sum()), "of", len(table))
table

## 3. The trace of one image

The flag with stars. The selector chooses `flag_stars`; the counter then finds the star shapes
and counts them.

In [ ]:
demo.trace_table(answers["flag_stars"])

## Reading the result

(to be written)

## Conclusion

(to be written)